# Correlation
## Example 1: is the population of children related to the population of women in a NYS county?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

nys_pop = pd.read_csv("../09-advanced_data_operations/data/ny_county_pop.csv")

nys_pop.head()


In [ ]:
children_pop = nys_pop.groupby("county", as_index=False)['child'].sum()
women_pop = nys_pop[nys_pop['gender'] == "F"].drop(columns='gender').set_index("county")['adult'].rename("women").reset_index()

compare_df = pd.merge(left=children_pop, right=women_pop, how="outer", on="county").fillna(0)

compare_df

### Compute correlation

In [ ]:
corr, p_value = pearsonr(compare_df["child"], compare_df["women"])

print("Correlation:", corr)
print("p-value:", p_value)

Interpretation: children population has a very strong relationship with women population. And such relationship is statistical significant.

### Visual inspection

In [ ]:
sns.set_style("whitegrid")
g=sns.regplot(data=compare_df, x="women", y="child")

g.text(x=1, y=550_000, s=f"{corr=:.3f}\n{p_value=:.3f}", fontsize=15)


## Example 2: is movie length related to rating?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

df = pd.read_csv("../08-core_data_manipulation/data/IMDB Top 250 Movies.csv")

df.head()


In [ ]:
df['run_time_timedelta'] = pd.to_timedelta(df['run_time'], unit='m', errors='coerce')

df['run_time_mins'] = df['run_time_timedelta'].dt.total_seconds()/60
df['box_office'] = pd.to_numeric(df['box_office'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

clean_df = df.dropna(subset=['run_time_mins', 'box_office', "rating"])

In [ ]:

corr, p_value = pearsonr(clean_df["run_time_mins"], clean_df["rating"])

print("Correlation:", corr)
print("p-value:", p_value)


Interpretation: The length of movie has a moderate and positive relationship with movie ratings.

### Visual inspection

In [ ]:
sns.set_style("whitegrid")
g=sns.regplot(data=clean_df, x="rating", y="run_time_mins")
g.text(x=8.8, y=225, s=f"{corr=:.3f}\n{p_value=:.3f}", fontsize=15)

## Example 3: is male-to-female ratio related to children population in NYS?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

nys_pop = pd.read_csv("../09-advanced_data_operations/data/ny_county_pop.csv")

nys_pop.head()

children_pop = nys_pop.groupby("county", as_index=False)['child'].sum()

compare_df = pd.merge(left=children_pop, right=nys_pop.sort_values(by='gender').set_index("gender").groupby("county", as_index=False).agg(
    male_female_rate = ('adult', lambda x: x.iloc[1]*100/x.iloc[0])
), on="county", how="left")

compare_df


In [ ]:
# compute correlation

corr, p_value = pearsonr(compare_df["male_female_rate"], compare_df['child'])

print("Correlation:", corr)
print("p-value:", p_value)


In [ ]:
g=sns.regplot(data=compare_df, x="child", y="male_female_rate")
g.text(x=400_000, y=120, s=f"{corr=:.3f}\n{p_value=:.3f}", fontsize=15)

### Interpretation

Male to female ratio is moderately negatively related to child population in a county in New York State, and such correlation is statistically significant.